In [3]:
import sys
print(sys.executable)
!{sys.executable} -m pip install lightgbm

c:\Users\atefe\.pyenv\pyenv-win\versions\3.11.3\python.exe


In [4]:
from lightgbm import LGBMRegressor

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm xgboost catboost seaborn

In [ ]:
# =========================================================
# 1. Imports
# =========================================================

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Display settings
sns.set_style("darkgrid", rc={"axes.facecolor": "white", "grid.color": ".8"})
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [ ]:
# =========================================================
# 2. Load data
# =========================================================

# Load the final feature dataset and parse the timestamp column as datetime
df = pd.read_csv("../data/df_features.csv", parse_dates=["timestamp"])

# Sort the data chronologically and reset the index
df = df.sort_values("timestamp").reset_index(drop=True)

# Show basic checks
print("Shape:", df.shape)
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
display(df.head(2))


In [ ]:
# =========================================================
# 3. Define the final feature set
# =========================================================

# Use all columns except timestamp and target as model features
feature_cols = [col for col in df.columns if col not in ["timestamp", "price"]]

# Show basic feature information
print("Number of selected features:", len(feature_cols))
print(feature_cols)

# Preview the selected feature columns
display(df[feature_cols].head(2))

In [ ]:
# =========================================================
# 4. Check the selected feature set
# =========================================================

# Show number of features
print("Number of selected features:", len(feature_cols))

# Show data type counts
display(df[feature_cols].dtypes.value_counts())

# Show missing values in the selected features
display(df[feature_cols].isna().sum().sort_values(ascending=False).head(20))

In [ ]:
# =========================================================
# 5. Create the hourly train, validation, and test split
# =========================================================

# Create chronological masks
train_mask = df["timestamp"] < "2025-01-01"
val_mask = (df["timestamp"] >= "2025-01-01") & (df["timestamp"] < "2025-07-01")
test_mask = df["timestamp"] >= "2025-07-01"

# Create split dataframes
train_df = df.loc[train_mask].copy()
val_df = df.loc[val_mask].copy()
test_df = df.loc[test_mask].copy()

# Show split shapes
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

# Show split date ranges
print("Train range:", train_df["timestamp"].min(), "to", train_df["timestamp"].max())
print("Validation range:", val_df["timestamp"].min(), "to", val_df["timestamp"].max())
print("Test range:", test_df["timestamp"].min(), "to", test_df["timestamp"].max())

In [ ]:
# =========================================================
# 6. Create sample weights for the training period
# =========================================================

# Create default training weights
train_weights = pd.Series(1.0, index=train_df.index)

# Assign regime-based weights
train_weights.loc[(train_df["timestamp"] >= "2019-01-01") & (train_df["timestamp"] < "2021-01-01")] = 0.6
train_weights.loc[(train_df["timestamp"] >= "2021-01-01") & (train_df["timestamp"] < "2023-01-01")] = 0.3
train_weights.loc[(train_df["timestamp"] >= "2023-01-01") & (train_df["timestamp"] < "2025-01-01")] = 1.0

# Show the weight distribution
display(train_weights.value_counts().sort_index())
display(train_weights.head())

In [ ]:
# =========================================================
# 7. Create hourly feature matrices and targets
# =========================================================

# Create feature matrices
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Create targets
y_train = train_df["price"].copy()
y_val = val_df["price"].copy()
y_test = test_df["price"].copy()

# Show shapes
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# =========================================================
# 8. Build the naive hourly benchmark
# =========================================================

# Define a function for the naive hourly forecast
def predict_price_lag_hourly(df_input, price_col="price", day_col="day_of_week"):
    # Create 24-hour and 168-hour lagged prices
    lag_1 = df_input[price_col].shift(24)
    lag_7 = df_input[price_col].shift(168)

    # Use D-1 for Tue, Wed, Thu, Fri, Sun and D-7 for Mon, Sat
    lag_1_days = {1, 2, 3, 4, 6}

    # Return the naive prediction series
    return pd.Series(
        np.where(df_input[day_col].isin(lag_1_days), lag_1, lag_7),
        index=df_input.index
    )

# Create naive hourly predictions for the full dataframe
df["price_pred_naive"] = predict_price_lag_hourly(df)

# Extract naive predictions for validation and test
y_val_naive = df.loc[val_df.index, "price_pred_naive"].copy()
y_test_naive = df.loc[test_df.index, "price_pred_naive"].copy()

In [ ]:
# =========================================================
# 9. Define evaluation metrics
# =========================================================

# Define hourly MAE
def mae_hourly(y_true, y_pred):
    # Compute the mean absolute error
    return mean_absolute_error(y_true, y_pred)

# Define daily average error based on hourly predictions
def dae_hourly(y_true, y_pred, timestamps):
    # Build a temporary dataframe with timestamps, true values, and predictions
    eval_df = pd.DataFrame({
        "timestamp": timestamps,
        "y_true": y_true,
        "y_pred": y_pred
    })

    # Aggregate to daily sums
    daily_eval = eval_df.groupby(eval_df["timestamp"].dt.floor("D")).agg({
        "y_true": "sum",
        "y_pred": "sum"
    })

    # Compute the mean daily average error
    return np.mean(np.abs(daily_eval["y_true"] - daily_eval["y_pred"]) / 24)

# Define relative MAE against the naive benchmark
def rmae(model_mae, naive_mae):
    # Compute relative MAE
    return model_mae / naive_mae

In [ ]:
# =========================================================
# 10. Evaluate the naive benchmark
# =========================================================

# Compute naive validation metrics
naive_val_mae = mae_hourly(y_val, y_val_naive)
naive_val_dae = dae_hourly(y_val, y_val_naive, val_df["timestamp"])

# Compute naive test metrics
naive_test_mae = mae_hourly(y_test, y_test_naive)
naive_test_dae = dae_hourly(y_test, y_test_naive, test_df["timestamp"])

# Print validation results
print("Naive validation MAE:", round(naive_val_mae, 4))
print("Naive validation DAE:", round(naive_val_dae, 4))

# Print test results
print("Naive test MAE:", round(naive_test_mae, 4))
print("Naive test DAE:", round(naive_test_dae, 4))

In [ ]:
# =========================================================
# 11. Build the Lasso baseline
# =========================================================

# Build a pipeline with feature scaling and Lasso
lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, max_iter=10000, random_state=42))
])

# Fit the Lasso model on the training data with sample weights
lasso_model.fit(X_train, y_train, model__sample_weight=train_weights)

# Predict on validation and test data
y_val_pred_lasso = pd.Series(lasso_model.predict(X_val), index=y_val.index)
y_test_pred_lasso = pd.Series(lasso_model.predict(X_test), index=y_test.index)

In [ ]:
# =========================================================
# 12. Evaluate the Lasso baseline
# =========================================================

# Compute Lasso validation metrics
lasso_val_mae = mae_hourly(y_val, y_val_pred_lasso)
lasso_val_dae = dae_hourly(y_val, y_val_pred_lasso, val_df["timestamp"])
lasso_val_rmae = rmae(lasso_val_mae, naive_val_mae)

# Compute Lasso test metrics
lasso_test_mae = mae_hourly(y_test, y_test_pred_lasso)
lasso_test_dae = dae_hourly(y_test, y_test_pred_lasso, test_df["timestamp"])
lasso_test_rmae = rmae(lasso_test_mae, naive_test_mae)

# Print validation results
print("Lasso validation MAE:", round(lasso_val_mae, 4))
print("Lasso validation DAE:", round(lasso_val_dae, 4))
print("Lasso validation RMAE:", round(lasso_val_rmae, 4))

# Print test results
print("Lasso test MAE:", round(lasso_test_mae, 4))
print("Lasso test DAE:", round(lasso_test_dae, 4))
print("Lasso test RMAE:", round(lasso_test_rmae, 4))

In [ ]:
# =========================================================
# 13. Tune the Lasso baseline on the validation set
# =========================================================

# Define candidate alpha values for Lasso tuning
lasso_param_grid = [0.001, 0.01, 0.1, 1.0]

# Store tuning results
lasso_tuning_results = []

# Initialize tracking variables
best_lasso_model = None
best_lasso_alpha = None
best_lasso_val_mae = np.inf
best_lasso_val_dae = np.inf

# Loop over candidate alpha values
for alpha in lasso_param_grid:
    # Build a Lasso pipeline with scaling
    lasso_candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=10000, random_state=42))
    ])

    # Fit on the training set
    lasso_candidate.fit(X_train, y_train, model__sample_weight=train_weights)

    # Predict on the validation set
    y_val_pred_candidate = pd.Series(lasso_candidate.predict(X_val), index=y_val.index)

    # Compute validation metrics
    val_mae = mae_hourly(y_val, y_val_pred_candidate)
    val_dae = dae_hourly(y_val, y_val_pred_candidate, val_df["timestamp"])
    val_rmae = rmae(val_mae, naive_val_mae)

    # Store results
    lasso_tuning_results.append({
        "alpha": alpha,
        "val_MAE": val_mae,
        "val_DAE": val_dae,
        "val_RMAE": val_rmae
    })

    # Update the best model based on validation MAE
    if val_mae < best_lasso_val_mae:
        best_lasso_val_mae = val_mae
        best_lasso_val_dae = val_dae
        best_lasso_alpha = alpha
        best_lasso_model = lasso_candidate

# Convert results to dataframe
lasso_tuning_results = pd.DataFrame(lasso_tuning_results).sort_values("val_MAE").reset_index(drop=True)

# Show tuning table
display(lasso_tuning_results.round(4))

# Print the best Lasso setting
print("Best Lasso alpha:", best_lasso_alpha)
print("Best Lasso validation MAE:", round(best_lasso_val_mae, 4))
print("Best Lasso validation DAE:", round(best_lasso_val_dae, 4))

In [ ]:
# =========================================================
# 14. Define hyperparameter grids for the tree-based models
# =========================================================

# Define compact hyperparameter grids for validation-based tuning
# These are kept moderate so the notebook remains practical to run

lgbm_param_grid = [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6, "num_leaves": 31, "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 350, "learning_rate": 0.05, "max_depth": 8, "num_leaves": 63, "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 8, "num_leaves": 63, "min_child_samples": 30, "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 350, "learning_rate": 0.10, "max_depth": 6, "num_leaves": 31, "min_child_samples": 30, "subsample": 0.9, "colsample_bytree": 0.8}
]

xgb_param_grid = [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6, "min_child_weight": 1, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
    {"n_estimators": 350, "learning_rate": 0.05, "max_depth": 8, "min_child_weight": 1, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 8, "min_child_weight": 3, "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 3.0},
    {"n_estimators": 350, "learning_rate": 0.10, "max_depth": 6, "min_child_weight": 3, "subsample": 0.9, "colsample_bytree": 0.8, "reg_lambda": 3.0}
]

catboost_param_grid = [
    {"iterations": 200, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 3},
    {"iterations": 350, "learning_rate": 0.05, "depth": 8, "l2_leaf_reg": 3},
    {"iterations": 500, "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 5},
    {"iterations": 350, "learning_rate": 0.10, "depth": 6, "l2_leaf_reg": 5}
]

# Show the number of parameter settings per model
print("LightGBM settings:", len(lgbm_param_grid))
print("XGBoost settings:", len(xgb_param_grid))
print("CatBoost settings:", len(catboost_param_grid))

In [ ]:
# =========================================================
# 15. Tune LightGBM on the validation set
# =========================================================

# Store tuning results
lgbm_tuning_results = []

# Initialize tracking variables
best_lgbm_model = None
best_lgbm_params = None
best_lgbm_val_mae = np.inf
best_lgbm_val_dae = np.inf

# Loop over candidate parameter settings
for params in lgbm_param_grid:
    # Build the LightGBM model
    lgbm_candidate = LGBMRegressor(
        objective="regression",
        random_state=42,
        verbose=-1,
        **params
    )

    # Fit on the training set
    lgbm_candidate.fit(X_train, y_train, sample_weight=train_weights)

    # Predict on the validation set
    y_val_pred_candidate = pd.Series(lgbm_candidate.predict(X_val), index=y_val.index)

    # Compute validation metrics
    val_mae = mae_hourly(y_val, y_val_pred_candidate)
    val_dae = dae_hourly(y_val, y_val_pred_candidate, val_df["timestamp"])
    val_rmae = rmae(val_mae, naive_val_mae)

    # Store results
    lgbm_tuning_results.append({
        "model": "LightGBM",
        **params,
        "val_MAE": val_mae,
        "val_DAE": val_dae,
        "val_RMAE": val_rmae
    })

    # Update the best model based on validation MAE
    if val_mae < best_lgbm_val_mae:
        best_lgbm_val_mae = val_mae
        best_lgbm_val_dae = val_dae
        best_lgbm_params = params
        best_lgbm_model = lgbm_candidate

# Convert results to dataframe
lgbm_tuning_results = pd.DataFrame(lgbm_tuning_results).sort_values("val_MAE").reset_index(drop=True)

# Show tuning table
display(lgbm_tuning_results.round(4))

# Print the best LightGBM setting
print("Best LightGBM params:", best_lgbm_params)
print("Best LightGBM validation MAE:", round(best_lgbm_val_mae, 4))
print("Best LightGBM validation DAE:", round(best_lgbm_val_dae, 4))

In [ ]:
# =========================================================
# 16. Tune XGBoost on the validation set
# =========================================================

# Store tuning results
xgb_tuning_results = []

# Initialize tracking variables
best_xgb_model = None
best_xgb_params = None
best_xgb_val_mae = np.inf
best_xgb_val_dae = np.inf

# Loop over candidate parameter settings
for params in xgb_param_grid:
    # Build the XGBoost model
    xgb_candidate = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        **params
    )

    # Fit on the training set
    xgb_candidate.fit(X_train, y_train, sample_weight=train_weights)

    # Predict on the validation set
    y_val_pred_candidate = pd.Series(xgb_candidate.predict(X_val), index=y_val.index)

    # Compute validation metrics
    val_mae = mae_hourly(y_val, y_val_pred_candidate)
    val_dae = dae_hourly(y_val, y_val_pred_candidate, val_df["timestamp"])
    val_rmae = rmae(val_mae, naive_val_mae)

    # Store results
    xgb_tuning_results.append({
        "model": "XGBoost",
        **params,
        "val_MAE": val_mae,
        "val_DAE": val_dae,
        "val_RMAE": val_rmae
    })

    # Update the best model based on validation MAE
    if val_mae < best_xgb_val_mae:
        best_xgb_val_mae = val_mae
        best_xgb_val_dae = val_dae
        best_xgb_params = params
        best_xgb_model = xgb_candidate

# Convert results to dataframe
xgb_tuning_results = pd.DataFrame(xgb_tuning_results).sort_values("val_MAE").reset_index(drop=True)

# Show tuning table
display(xgb_tuning_results.round(4))

# Print the best XGBoost setting
print("Best XGBoost params:", best_xgb_params)
print("Best XGBoost validation MAE:", round(best_xgb_val_mae, 4))
print("Best XGBoost validation DAE:", round(best_xgb_val_dae, 4))

In [ ]:
# =========================================================
# 17. Tune CatBoost on the validation set
# =========================================================

# Store tuning results
catboost_tuning_results = []

# Initialize tracking variables
best_catboost_model = None
best_catboost_params = None
best_catboost_val_mae = np.inf
best_catboost_val_dae = np.inf

# Loop over candidate parameter settings
for params in catboost_param_grid:
    # Build the CatBoost model
    catboost_candidate = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        verbose=0,
        random_state=42,
        **params
    )

    # Fit on the training set
    catboost_candidate.fit(X_train, y_train, sample_weight=train_weights)

    # Predict on the validation set
    y_val_pred_candidate = pd.Series(catboost_candidate.predict(X_val), index=y_val.index)

    # Compute validation metrics
    val_mae = mae_hourly(y_val, y_val_pred_candidate)
    val_dae = dae_hourly(y_val, y_val_pred_candidate, val_df["timestamp"])
    val_rmae = rmae(val_mae, naive_val_mae)

    # Store results
    catboost_tuning_results.append({
        "model": "CatBoost",
        **params,
        "val_MAE": val_mae,
        "val_DAE": val_dae,
        "val_RMAE": val_rmae
    })

    # Update the best model based on validation MAE
    if val_mae < best_catboost_val_mae:
        best_catboost_val_mae = val_mae
        best_catboost_val_dae = val_dae
        best_catboost_params = params
        best_catboost_model = catboost_candidate

# Convert results to dataframe
catboost_tuning_results = pd.DataFrame(catboost_tuning_results).sort_values("val_MAE").reset_index(drop=True)

# Show tuning table
display(catboost_tuning_results.round(4))

# Print the best CatBoost setting
print("Best CatBoost params:", best_catboost_params)
print("Best CatBoost validation MAE:", round(best_catboost_val_mae, 4))
print("Best CatBoost validation DAE:", round(best_catboost_val_dae, 4))

In [ ]:
# =========================================================
# 18. Compare validation results across all tuned models
# =========================================================

# Build a combined validation comparison table
validation_results = pd.DataFrame([
    {
        "model": "Naive",
        "val_MAE": naive_val_mae,
        "val_DAE": naive_val_dae,
        "val_RMAE": 1.0
    },
    {
        "model": "Lasso_fixed",
        "val_MAE": lasso_val_mae,
        "val_DAE": lasso_val_dae,
        "val_RMAE": lasso_val_rmae
    },
    {
        "model": "Lasso_tuned",
        "val_MAE": best_lasso_val_mae,
        "val_DAE": best_lasso_val_dae,
        "val_RMAE": rmae(best_lasso_val_mae, naive_val_mae)
    },
    {
        "model": "LightGBM",
        "val_MAE": best_lgbm_val_mae,
        "val_DAE": best_lgbm_val_dae,
        "val_RMAE": rmae(best_lgbm_val_mae, naive_val_mae)
    },
    {
        "model": "XGBoost",
        "val_MAE": best_xgb_val_mae,
        "val_DAE": best_xgb_val_dae,
        "val_RMAE": rmae(best_xgb_val_mae, naive_val_mae)
    },
    {
        "model": "CatBoost",
        "val_MAE": best_catboost_val_mae,
        "val_DAE": best_catboost_val_dae,
        "val_RMAE": rmae(best_catboost_val_mae, naive_val_mae)
    }
]).sort_values("val_MAE").reset_index(drop=True)

# Display the validation comparison table
display(validation_results.round(4))

In [ ]:
# =========================================================
# 18.1 Compare validation and test results of the current models
# =========================================================

# Build a comparison table for validation and test performance
results_stage_18_1 = pd.DataFrame([
    {
        "model": "Naive",
        "val_MAE": naive_val_mae,
        "val_DAE": naive_val_dae,
        "test_MAE": naive_test_mae,
        "test_DAE": naive_test_dae,
        "val_RMAE": 1.0,
        "test_RMAE": 1.0
    },
    {
        "model": "Lasso_fixed",
        "val_MAE": lasso_val_mae,
        "val_DAE": lasso_val_dae,
        "test_MAE": lasso_test_mae,
        "test_DAE": lasso_test_dae,
        "val_RMAE": lasso_val_rmae,
        "test_RMAE": lasso_test_rmae
    },
    {
        "model": "Lasso_tuned",
        "val_MAE": best_lasso_val_mae,
        "val_DAE": best_lasso_val_dae,
        "test_MAE": np.nan,
        "test_DAE": np.nan,
        "val_RMAE": rmae(best_lasso_val_mae, naive_val_mae),
        "test_RMAE": np.nan
    },
    {
        "model": "LightGBM",
        "val_MAE": best_lgbm_val_mae,
        "val_DAE": best_lgbm_val_dae,
        "test_MAE": np.nan,
        "test_DAE": np.nan,
        "val_RMAE": rmae(best_lgbm_val_mae, naive_val_mae),
        "test_RMAE": np.nan
    },
    {
        "model": "XGBoost",
        "val_MAE": best_xgb_val_mae,
        "val_DAE": best_xgb_val_dae,
        "test_MAE": np.nan,
        "test_DAE": np.nan,
        "val_RMAE": rmae(best_xgb_val_mae, naive_val_mae),
        "test_RMAE": np.nan
    },
    {
        "model": "CatBoost",
        "val_MAE": best_catboost_val_mae,
        "val_DAE": best_catboost_val_dae,
        "test_MAE": np.nan,
        "test_DAE": np.nan,
        "val_RMAE": rmae(best_catboost_val_mae, naive_val_mae),
        "test_RMAE": np.nan
    }
]).round(4)

# Show the comparison table
display(results_stage_18_1)

In [ ]:
# =========================================================
# 19. Plot LightGBM feature importance
# =========================================================

# Build a feature-importance table from the best LightGBM model
lgbm_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_gain": best_lgbm_model.booster_.feature_importance(importance_type="gain"),
    "importance_split": best_lgbm_model.booster_.feature_importance(importance_type="split")
}).sort_values("importance_gain", ascending=False).reset_index(drop=True)

# Show the ranked importance table
display(lgbm_importance)

# Plot the top 20 features by gain
plt.figure(figsize=(10, 6))
plt.barh(
    lgbm_importance["feature"].head(20)[::-1],
    lgbm_importance["importance_gain"].head(20)[::-1]
)
plt.xlabel("LightGBM importance (gain)")
plt.ylabel("Feature")
plt.title("Top 20 LightGBM feature importances")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import sklearn

print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
import sys
!{sys.executable} -m pip show numpy scikit-learn shap

In [ ]:
import sys
!{sys.executable} -m pip install --force-reinstall --no-cache-dir lightgbm xgboost catboost

In [ ]:
# =========================================================
# 20. Compute permutation importance for the best LightGBM model
# =========================================================

from sklearn.inspection import permutation_importance

# Compute permutation importance on the validation set
perm_result = permutation_importance(
    best_lgbm_model,
    X_val,
    y_val,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=1
)

# Build a results table
perm_importance = pd.DataFrame({
    "feature": X_val.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

# Show the ranked permutation importance table
display(perm_importance)

In [ ]:
# =========================================================
# 21. Show the weakest features from permutation importance
# =========================================================

# Show the 10 least important features
least_important_features = perm_importance.sort_values("importance_mean", ascending=True).reset_index(drop=True)

display(least_important_features.head(10))

# Print only the feature names for easy review
print("Weakest features:")
print(least_important_features.head(10)["feature"].tolist())

In [ ]:
# =========================================================
# 22. Define a reduced feature set for a first pruning test
# =========================================================

# Remove the strongest conservative pruning candidates
features_to_remove = [
    "net_export_lag_24h",
    "coal_price_lag_24h",
    "total_generation_lag_168h",
    "price_rolling_168h",
    "year",
    "is_crisis_period"
]

# Build the reduced feature list
feature_cols_reduced = [col for col in feature_cols if col not in features_to_remove]

# Show the result
print("Original number of features:", len(feature_cols))
print("Reduced number of features:", len(feature_cols_reduced))
print("Removed features:", features_to_remove)

In [ ]:
# =========================================================
# 23. Create reduced feature matrices
# =========================================================

# Create reduced feature matrices
X_train_red = train_df[feature_cols_reduced].copy()
X_val_red = val_df[feature_cols_reduced].copy()
X_test_red = test_df[feature_cols_reduced].copy()

# Show shapes
print("X_train_red:", X_train_red.shape)
print("X_val_red:", X_val_red.shape)
print("X_test_red:", X_test_red.shape)

In [ ]:
# =========================================================
# 24. Refit the best LightGBM on the reduced feature set
# =========================================================

# Build the reduced LightGBM model using the previously best parameters
lgbm_reduced = LGBMRegressor(
    objective="regression",
    random_state=42,
    verbose=-1,
    **best_lgbm_params
)

# Fit on the training set
lgbm_reduced.fit(X_train_red, y_train, sample_weight=train_weights)

# Predict on the validation set
y_val_pred_lgbm_red = pd.Series(lgbm_reduced.predict(X_val_red), index=y_val.index)

# Compute validation metrics
lgbm_red_val_mae = mae_hourly(y_val, y_val_pred_lgbm_red)
lgbm_red_val_dae = dae_hourly(y_val, y_val_pred_lgbm_red, val_df["timestamp"])
lgbm_red_val_rmae = rmae(lgbm_red_val_mae, naive_val_mae)

# Print validation results
print("Reduced LightGBM validation MAE:", round(lgbm_red_val_mae, 4))
print("Reduced LightGBM validation DAE:", round(lgbm_red_val_dae, 4))
print("Reduced LightGBM validation RMAE:", round(lgbm_red_val_rmae, 4))

In [ ]:
# =========================================================
# 25. Compare original and reduced LightGBM validation results
# =========================================================

# Build a comparison table
lgbm_feature_pruning_comparison = pd.DataFrame({
    "model_version": ["LightGBM_original", "LightGBM_reduced"],
    "val_MAE": [best_lgbm_val_mae, lgbm_red_val_mae],
    "val_DAE": [best_lgbm_val_dae, lgbm_red_val_dae],
    "val_RMAE": [
        rmae(best_lgbm_val_mae, naive_val_mae),
        lgbm_red_val_rmae
    ]
}).round(4)

# Show the comparison
display(lgbm_feature_pruning_comparison)

In [ ]:
# =========================================================
# 26. Refit the fixed Lasso baseline on the reduced feature set
# =========================================================

# Build a pipeline with feature scaling and Lasso
lasso_model_red = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, max_iter=10000, random_state=42))
])

# Fit the Lasso model on the reduced training data
lasso_model_red.fit(X_train_red, y_train, model__sample_weight=train_weights)

# Predict on the reduced validation set
y_val_pred_lasso_red = pd.Series(lasso_model_red.predict(X_val_red), index=y_val.index)

# Compute validation metrics
lasso_red_val_mae = mae_hourly(y_val, y_val_pred_lasso_red)
lasso_red_val_dae = dae_hourly(y_val, y_val_pred_lasso_red, val_df["timestamp"])
lasso_red_val_rmae = rmae(lasso_red_val_mae, naive_val_mae)

# Print validation results
print("Reduced Lasso validation MAE:", round(lasso_red_val_mae, 4))
print("Reduced Lasso validation DAE:", round(lasso_red_val_dae, 4))
print("Reduced Lasso validation RMAE:", round(lasso_red_val_rmae, 4))

In [ ]:
# =========================================================
# 27. Refit the best XGBoost model on the reduced feature set
# =========================================================

# Build the reduced XGBoost model using the previously best parameters
xgb_reduced = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **best_xgb_params
)

# Fit on the reduced training set
xgb_reduced.fit(X_train_red, y_train, sample_weight=train_weights)

# Predict on the reduced validation set
y_val_pred_xgb_red = pd.Series(xgb_reduced.predict(X_val_red), index=y_val.index)

# Compute validation metrics
xgb_red_val_mae = mae_hourly(y_val, y_val_pred_xgb_red)
xgb_red_val_dae = dae_hourly(y_val, y_val_pred_xgb_red, val_df["timestamp"])
xgb_red_val_rmae = rmae(xgb_red_val_mae, naive_val_mae)

# Print validation results
print("Reduced XGBoost validation MAE:", round(xgb_red_val_mae, 4))
print("Reduced XGBoost validation DAE:", round(xgb_red_val_dae, 4))
print("Reduced XGBoost validation RMAE:", round(xgb_red_val_rmae, 4))

In [ ]:
# =========================================================
# 28. Refit the best CatBoost model on the reduced feature set
# =========================================================

# Build the reduced CatBoost model using the previously best parameters
catboost_reduced = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    verbose=0,
    random_state=42,
    **best_catboost_params
)

# Fit on the reduced training set
catboost_reduced.fit(X_train_red, y_train, sample_weight=train_weights)

# Predict on the reduced validation set
y_val_pred_catboost_red = pd.Series(catboost_reduced.predict(X_val_red), index=y_val.index)

# Compute validation metrics
catboost_red_val_mae = mae_hourly(y_val, y_val_pred_catboost_red)
catboost_red_val_dae = dae_hourly(y_val, y_val_pred_catboost_red, val_df["timestamp"])
catboost_red_val_rmae = rmae(catboost_red_val_mae, naive_val_mae)

# Print validation results
print("Reduced CatBoost validation MAE:", round(catboost_red_val_mae, 4))
print("Reduced CatBoost validation DAE:", round(catboost_red_val_dae, 4))
print("Reduced CatBoost validation RMAE:", round(catboost_red_val_rmae, 4))

In [ ]:
# =========================================================
# 29. Compare all original and reduced validation results
# =========================================================

# Build a comparison table
validation_results_reduced = pd.DataFrame([
    {"model": "Naive", "val_MAE": naive_val_mae, "val_DAE": naive_val_dae, "val_RMAE": 1.0},
    {"model": "Lasso_original", "val_MAE": lasso_val_mae, "val_DAE": lasso_val_dae, "val_RMAE": lasso_val_rmae},
    {"model": "Lasso_reduced", "val_MAE": lasso_red_val_mae, "val_DAE": lasso_red_val_dae, "val_RMAE": lasso_red_val_rmae},
    {"model": "LightGBM_original", "val_MAE": best_lgbm_val_mae, "val_DAE": best_lgbm_val_dae, "val_RMAE": rmae(best_lgbm_val_mae, naive_val_mae)},
    {"model": "LightGBM_reduced", "val_MAE": lgbm_red_val_mae, "val_DAE": lgbm_red_val_dae, "val_RMAE": lgbm_red_val_rmae},
    {"model": "XGBoost_original", "val_MAE": best_xgb_val_mae, "val_DAE": best_xgb_val_dae, "val_RMAE": rmae(best_xgb_val_mae, naive_val_mae)},
    {"model": "XGBoost_reduced", "val_MAE": xgb_red_val_mae, "val_DAE": xgb_red_val_dae, "val_RMAE": xgb_red_val_rmae},
    {"model": "CatBoost_original", "val_MAE": best_catboost_val_mae, "val_DAE": best_catboost_val_dae, "val_RMAE": rmae(best_catboost_val_mae, naive_val_mae)},
    {"model": "CatBoost_reduced", "val_MAE": catboost_red_val_mae, "val_DAE": catboost_red_val_dae, "val_RMAE": catboost_red_val_rmae}
]).sort_values("val_MAE").reset_index(drop=True)

# Show the comparison
display(validation_results_reduced.round(4))

In [ ]:
# =========================================================
# 30. Build XGBoost feature-importance table
# =========================================================

# Get the fitted XGBoost booster
xgb_booster = xgb_reduced.get_booster()

# Get feature importance dictionaries from XGBoost
xgb_gain = xgb_booster.get_score(importance_type="gain")
xgb_weight = xgb_booster.get_score(importance_type="weight")
xgb_cover = xgb_booster.get_score(importance_type="cover")
xgb_total_gain = xgb_booster.get_score(importance_type="total_gain")

# Build a complete importance table for all reduced features
xgb_importance = pd.DataFrame({
    "feature": feature_cols_reduced,
    "importance_gain": [xgb_gain.get(f, 0.0) for f in feature_cols_reduced],
    "importance_weight": [xgb_weight.get(f, 0.0) for f in feature_cols_reduced],
    "importance_cover": [xgb_cover.get(f, 0.0) for f in feature_cols_reduced],
    "importance_total_gain": [xgb_total_gain.get(f, 0.0) for f in feature_cols_reduced]
}).sort_values("importance_gain", ascending=False).reset_index(drop=True)

# Show the ranked importance table
display(xgb_importance)

In [ ]:
# =========================================================
# 33. Define a second reduced feature set for XGBoost
# =========================================================

# Remove a small second batch of weak remaining features
features_to_remove_2 = [
    #"renewable_ramp",
    #"wind_offshore",
    "total_generation_lag_24h",
    "net_export_lag_168h"
]

# Build the second reduced feature list
feature_cols_reduced_2 = [col for col in feature_cols_reduced if col not in features_to_remove_2]

# Show the result
print("First reduced number of features:", len(feature_cols_reduced))
print("Second reduced number of features:", len(feature_cols_reduced_2))
print("Removed in second round:", features_to_remove_2)

In [ ]:
# =========================================================
# 34. Create the second reduced feature matrices
# =========================================================

# Create second reduced feature matrices
X_train_red2 = train_df[feature_cols_reduced_2].copy()
X_val_red2 = val_df[feature_cols_reduced_2].copy()
X_test_red2 = test_df[feature_cols_reduced_2].copy()

# Show shapes
print("X_train_red2:", X_train_red2.shape)
print("X_val_red2:", X_val_red2.shape)
print("X_test_red2:", X_test_red2.shape)

In [ ]:
# =========================================================
# 35. Refit the best XGBoost on the second reduced feature set
# =========================================================

# Build the second reduced XGBoost model using the previously best parameters
xgb_reduced_2 = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **best_xgb_params
)

# Fit on the second reduced training set
xgb_reduced_2.fit(X_train_red2, y_train, sample_weight=train_weights)

# Predict on the second reduced validation set
y_val_pred_xgb_red2 = pd.Series(xgb_reduced_2.predict(X_val_red2), index=y_val.index)

# Compute validation metrics
xgb_red2_val_mae = mae_hourly(y_val, y_val_pred_xgb_red2)
xgb_red2_val_dae = dae_hourly(y_val, y_val_pred_xgb_red2, val_df["timestamp"])
xgb_red2_val_rmae = rmae(xgb_red2_val_mae, naive_val_mae)

# Print validation results
print("Second reduced XGBoost validation MAE:", round(xgb_red2_val_mae, 4))
print("Second reduced XGBoost validation DAE:", round(xgb_red2_val_dae, 4))
print("Second reduced XGBoost validation RMAE:", round(xgb_red2_val_rmae, 4))

In [ ]:
# =========================================================
# 36. Compare original and pruned XGBoost validation results
# =========================================================

# Build a comparison table
xgb_pruning_comparison = pd.DataFrame({
    "model_version": [
        "XGBoost_original",
        "XGBoost_reduced_1",
        "XGBoost_reduced_2"
    ],
    "val_MAE": [
        best_xgb_val_mae,
        xgb_red_val_mae,
        xgb_red2_val_mae
    ],
    "val_DAE": [
        best_xgb_val_dae,
        xgb_red_val_dae,
        xgb_red2_val_dae
    ],
    "val_RMAE": [
        rmae(best_xgb_val_mae, naive_val_mae),
        xgb_red_val_rmae,
        xgb_red2_val_rmae
    ]
}).round(4)

# Show the comparison
display(xgb_pruning_comparison)

In [ ]:
# =========================================================
# 37. Set the final chosen feature set
# =========================================================

# Use the first reduced feature set as the final selected feature set
final_feature_cols = feature_cols_reduced.copy()

# Show final feature count and names
print("Final number of features:", len(final_feature_cols))
print(final_feature_cols)

In [ ]:
# =========================================================
# 38. Create final train and test matrices
# =========================================================

# Combine train and validation data for final model training
train_val_df = pd.concat([train_df, val_df], axis=0)

# Create final feature matrices
X_final_train = train_val_df[final_feature_cols].copy()
X_final_test = test_df[final_feature_cols].copy()

# Create final targets
y_final_train = train_val_df["price"].copy()
y_final_test = test_df["price"].copy()

# Create final sample weights
val_weights = pd.Series(1.0, index=val_df.index)
final_train_weights = pd.concat([train_weights, val_weights], axis=0)

# Show shapes
print("X_final_train:", X_final_train.shape)
print("X_final_test:", X_final_test.shape)
print("y_final_train:", y_final_train.shape)
print("y_final_test:", y_final_test.shape)

In [ ]:
# =========================================================
# 39. Fit the final fixed Lasso model on the chosen feature set
# =========================================================

# Build a pipeline with feature scaling and Lasso
lasso_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, max_iter=10000, random_state=42))
])

# Fit the final Lasso model on train + validation
lasso_final.fit(X_final_train, y_final_train, model__sample_weight=final_train_weights)

# Predict on the test set
y_test_pred_lasso_final = pd.Series(lasso_final.predict(X_final_test), index=y_final_test.index)

In [ ]:
# =========================================================
# 40. Evaluate the final fixed Lasso model
# =========================================================

# Compute final Lasso test metrics
lasso_final_test_mae = mae_hourly(y_final_test, y_test_pred_lasso_final)
lasso_final_test_dae = dae_hourly(y_final_test, y_test_pred_lasso_final, test_df["timestamp"])
lasso_final_test_rmae = rmae(lasso_final_test_mae, naive_test_mae)

# Print final Lasso results
print("Final Lasso test MAE:", round(lasso_final_test_mae, 4))
print("Final Lasso test DAE:", round(lasso_final_test_dae, 4))
print("Final Lasso test RMAE:", round(lasso_final_test_rmae, 4))

In [ ]:
# =========================================================
# 41. Fit the final LightGBM model on the chosen feature set
# =========================================================

# Build the final LightGBM model with the best validation parameters
lgbm_final = LGBMRegressor(
    objective="regression",
    random_state=42,
    verbose=-1,
    **best_lgbm_params
)

# Fit the final LightGBM model on train + validation
lgbm_final.fit(X_final_train, y_final_train, sample_weight=final_train_weights)

# Predict on the test set
y_test_pred_lgbm_final = pd.Series(lgbm_final.predict(X_final_test), index=y_final_test.index)

In [ ]:
# =========================================================
# 42. Evaluate the final LightGBM model
# =========================================================

# Compute final LightGBM test metrics
lgbm_final_test_mae = mae_hourly(y_final_test, y_test_pred_lgbm_final)
lgbm_final_test_dae = dae_hourly(y_final_test, y_test_pred_lgbm_final, test_df["timestamp"])
lgbm_final_test_rmae = rmae(lgbm_final_test_mae, naive_test_mae)

# Print final LightGBM results
print("Final LightGBM test MAE:", round(lgbm_final_test_mae, 4))
print("Final LightGBM test DAE:", round(lgbm_final_test_dae, 4))
print("Final LightGBM test RMAE:", round(lgbm_final_test_rmae, 4))


In [ ]:
# =========================================================
# 43. Fit the final XGBoost model on the chosen feature set
# =========================================================

# Build the final XGBoost model with the best validation parameters
xgb_final = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **best_xgb_params
)

# Fit the final XGBoost model on train + validation
xgb_final.fit(X_final_train, y_final_train, sample_weight=final_train_weights)

# Predict on the test set
y_test_pred_xgb_final = pd.Series(xgb_final.predict(X_final_test), index=y_final_test.index)

In [ ]:
# =========================================================
# 44. Evaluate the final XGBoost model
# =========================================================

# Compute final XGBoost test metrics
xgb_final_test_mae = mae_hourly(y_final_test, y_test_pred_xgb_final)
xgb_final_test_dae = dae_hourly(y_final_test, y_test_pred_xgb_final, test_df["timestamp"])
xgb_final_test_rmae = rmae(xgb_final_test_mae, naive_test_mae)

# Print final XGBoost results
print("Final XGBoost test MAE:", round(xgb_final_test_mae, 4))
print("Final XGBoost test DAE:", round(xgb_final_test_dae, 4))
print("Final XGBoost test RMAE:", round(xgb_final_test_rmae, 4))

In [ ]:
# =========================================================
# 45. Fit the final CatBoost model on the chosen feature set
# =========================================================

# Build the final CatBoost model with the best validation parameters
catboost_final = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    verbose=0,
    random_state=42,
    **best_catboost_params
)

# Fit the final CatBoost model on train + validation
catboost_final.fit(X_final_train, y_final_train, sample_weight=final_train_weights)

# Predict on the test set
y_test_pred_catboost_final = pd.Series(catboost_final.predict(X_final_test), index=y_final_test.index)

In [ ]:
# =========================================================
# 46. Evaluate the final CatBoost model
# =========================================================

# Compute final CatBoost test metrics
catboost_final_test_mae = mae_hourly(y_final_test, y_test_pred_catboost_final)
catboost_final_test_dae = dae_hourly(y_final_test, y_test_pred_catboost_final, test_df["timestamp"])
catboost_final_test_rmae = rmae(catboost_final_test_mae, naive_test_mae)

# Print final CatBoost results
print("Final CatBoost test MAE:", round(catboost_final_test_mae, 4))
print("Final CatBoost test DAE:", round(catboost_final_test_dae, 4))
print("Final CatBoost test RMAE:", round(catboost_final_test_rmae, 4))

In [ ]:
# =========================================================
# 47. Final comparison table on the test set
# =========================================================

# Create the final comparison table
results_final = pd.DataFrame({
    "model": ["Naive", "Lasso_Final", "LightGBM_Final", "XGBoost_Final", "CatBoost_Final"],
    "test_MAE": [
        naive_test_mae,
        lasso_final_test_mae,
        lgbm_final_test_mae,
        xgb_final_test_mae,
        catboost_final_test_mae
    ],
    "test_DAE": [
        naive_test_dae,
        lasso_final_test_dae,
        lgbm_final_test_dae,
        xgb_final_test_dae,
        catboost_final_test_dae
    ],
    "test_RMAE": [
        1.0,
        lasso_final_test_rmae,
        lgbm_final_test_rmae,
        xgb_final_test_rmae,
        catboost_final_test_rmae
    ]
}).sort_values("test_MAE").reset_index(drop=True).round(4)

# Display the final comparison table
display(results_final)

In [ ]:
final_feature_cols = feature_cols_reduced.copy()

In [ ]:
# Print the number of final selected feature columns
print("Number of final features:", len(final_feature_cols))

# Print the final selected feature columns
for col in final_feature_cols:
    print(col)

In [ ]:
"""
final_feature_cols = [
    "load",
    "wind_offshore",
    "wind_onshore",
    "solar",
    "hour",
    "day_of_week",
    "month",
    "temperature",
    "wind_speed",
    "is_weekend",
    "price_lag_24h",
    "price_lag_168h",
    "price_rolling_24h",
    "is_holiday",
    "is_hol_or_week",
    "price_volatility_24h",
    "gas_price_lag_24h",
    "gas_price_lag_168h",
    "coal_price_lag_168h",
    "co2_price_lag_24h",
    "total_generation_lag_24h",
    "net_export_lag_168h",
    "is_peak_hour",
    "wind_x_peak",
    "solar_x_demand",
    "residual_load",
    "load_ramp",
    "renewable_ramp",
    "total_wind_forecast",
    "delta_wind_forecast"
]
"""

In [ ]:
# =========================================================
# 48. Define a richer local CatBoost grid on the reduced feature set
# =========================================================

catboost_param_grid_red = [
    {"iterations": 300, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 3},
    {"iterations": 500, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 3},
    {"iterations": 700, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 5},

    {"iterations": 300, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 3},
    {"iterations": 500, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 3},
    {"iterations": 700, "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 5},

    {"iterations": 300, "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 3},
    {"iterations": 500, "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 5},
    {"iterations": 700, "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 7},

    {"iterations": 300, "learning_rate": 0.05, "depth": 8, "l2_leaf_reg": 3},
    {"iterations": 500, "learning_rate": 0.05, "depth": 8, "l2_leaf_reg": 5},
    {"iterations": 700, "learning_rate": 0.05, "depth": 8, "l2_leaf_reg": 7}
]

print("Number of CatBoost settings:", len(catboost_param_grid_red))

In [ ]:
# =========================================================
# 49. Retune CatBoost on the reduced feature set
# =========================================================

catboost_tuning_results_red = []

best_catboost_model_red = None
best_catboost_params_red = None
best_catboost_val_mae_red = np.inf
best_catboost_val_dae_red = np.inf

for params in catboost_param_grid_red:
    catboost_candidate_red = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        verbose=0,
        random_state=42,
        **params
    )

    catboost_candidate_red.fit(
        X_train_red,
        y_train,
        sample_weight=train_weights
    )

    y_val_pred_candidate = pd.Series(
        catboost_candidate_red.predict(X_val_red),
        index=y_val.index
    )

    val_mae = mae_hourly(y_val, y_val_pred_candidate)
    val_dae = dae_hourly(y_val, y_val_pred_candidate, val_df["timestamp"])
    val_rmae = rmae(val_mae, naive_val_mae)

    catboost_tuning_results_red.append({
        **params,
        "val_MAE": val_mae,
        "val_DAE": val_dae,
        "val_RMAE": val_rmae
    })

    if val_mae < best_catboost_val_mae_red:
        best_catboost_val_mae_red = val_mae
        best_catboost_val_dae_red = val_dae
        best_catboost_params_red = params
        best_catboost_model_red = catboost_candidate_red

catboost_tuning_results_red = (
    pd.DataFrame(catboost_tuning_results_red)
    .sort_values("val_MAE")
    .reset_index(drop=True)
)

display(catboost_tuning_results_red.round(4))

print("Best reduced CatBoost params:", best_catboost_params_red)
print("Best reduced CatBoost validation MAE:", round(best_catboost_val_mae_red, 4))
print("Best reduced CatBoost validation DAE:", round(best_catboost_val_dae_red, 4))

In [ ]:
# =========================================================
# 50. Fit the retuned final CatBoost on train + validation
# =========================================================

catboost_final_retuned = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    verbose=0,
    random_state=42,
    **best_catboost_params_red
)

catboost_final_retuned.fit(
    X_final_train,
    y_final_train,
    sample_weight=final_train_weights
)

y_test_pred_catboost_final_retuned = pd.Series(
    catboost_final_retuned.predict(X_final_test),
    index=y_final_test.index
)

In [ ]:
# =========================================================
# 51. Evaluate the retuned final CatBoost on the test set
# =========================================================

catboost_final_retuned_test_mae = mae_hourly(y_final_test, y_test_pred_catboost_final_retuned)
catboost_final_retuned_test_dae = dae_hourly(y_final_test, y_test_pred_catboost_final_retuned, test_df["timestamp"])
catboost_final_retuned_test_rmae = rmae(catboost_final_retuned_test_mae, naive_test_mae)

print("Retuned CatBoost test MAE:", round(catboost_final_retuned_test_mae, 4))
print("Retuned CatBoost test DAE:", round(catboost_final_retuned_test_dae, 4))
print("Retuned CatBoost test RMAE:", round(catboost_final_retuned_test_rmae, 4))